# Simulated Data Experiments

This notebook contains the experiments for *Multiple hypothesis testing with simulated data*.

- Helper functions are defined in the first section to run the experiments.
- For each experiment presented in the paper, a dedicated cell is provided below.
- Running a cell will execute the experiment and automatically save both the results and the corresponding figures.

## Helper functions

In [1]:
import os
import numpy as np
import pandas as pd
from scipy.stats import binomtest, binom
import random
from utils_plot import load_and_plot, get_run_description

def run_test(n_successes, n, p=0.5, U=None):
    p_value_ = binomtest(n_successes, n=n, p=p, alternative='greater').pvalue
    boundary_mass = binom.pmf(n_successes, n, p)
    p_value = p_value_ - boundary_mass + U * boundary_mass
    return p_value


def analyze_performance(rejections, test_y):
    n_outliers = np.sum(test_y == 1)
    n_false_discoveries = np.sum(test_y[rejections] == 0)
    n_true_discoveries = np.sum(test_y[rejections] == 1)
    assert n_true_discoveries + n_false_discoveries == len(rejections)
    power = n_true_discoveries / n_outliers if n_outliers else 1
    fdr = n_false_discoveries / (n_false_discoveries + n_true_discoveries) if (n_false_discoveries + n_true_discoveries) > 0 else 0
    type1 = n_false_discoveries / (len(test_y) - n_outliers)
    fwer_ = int(n_false_discoveries > 0)
    return power, type1, fwer_, fdr


def BH(pvalues, level):
    """
    Benjamini-Hochberg procedure.
    """
    n = len(pvalues)
    pvalues_sort_ind = np.argsort(pvalues)
    pvalues_sort = np.sort(pvalues) #p(1) < p(2) < .... < p(n)

    comp = pvalues_sort <= (level* np.arange(1,n+1)/n)
    #get first location i0 at which p(k) <= level * k / n
    comp = comp[::-1]
    comp_true_ind = np.nonzero(comp)[0]
    i0 = comp_true_ind[0] if comp_true_ind.size > 0 else n
    nb_rej = n - i0
    threshold = pvalues[pvalues_sort_ind[nb_rej - 1]]
    return pvalues_sort_ind[:nb_rej], threshold


def synth_powered_BH(pvalues_real, pvalues_synth_powered, level, epsilon):
    """
    synthetic-powered Benjamini-Hochberg procedure.
    """
    p = np.asarray(pvalues_real)
    p_synth = np.asarray(pvalues_synth_powered)
    m = len(p)

    k_vals = np.arange(1, m + 1)
    adj = (k_vals * epsilon / m)[:, None]  # shape (m, 1)

    # Compute \tilde{p}_{k,j} = min{p_j, max{p^synth_j, p_j - k*epsilon/m}}
    P_tilde = np.minimum(p, np.maximum(p_synth, p - adj))  # shape (m, m)

    # Compute the k-th smallest element for each row (using partition or full sort)
    p_kth = np.partition(P_tilde, k_vals - 1, axis=1)[np.arange(m), k_vals - 1]
    # p_kth = np.sort(P_tilde, axis=1)[np.arange(m), k_vals - 1]

    thresholds = level * k_vals / m
    cond = p_kth <= thresholds

    if np.any(cond):
        k_star = np.max(np.where(cond)[0]) + 1  # +1 for 1-based indexing
        # Recompute \tilde{p} for k_star only
        P_tilde_star = np.minimum(p, np.maximum(p_synth, p - k_star * epsilon / m))
        threshold = np.partition(P_tilde_star, k_star - 1)[k_star - 1]
        rejections = np.where(P_tilde_star <= threshold)[0]
    else:
        k_star, threshold = 0, 0
        rejections = np.array([], dtype=int)

    return rejections, threshold


def  generate_data_and_return_p_values(n_real, n_synth, y_test, prob_alternative, prob_null_synth, prob_alternative_synth):
    real_pvalues, synth_pvalues, synth_powered_pvalues = [], [], []
    n_test = len(y_test)
    probs_real = y_test * prob_alternative + (1-y_test) * 0.5
    probs_synth = y_test * prob_alternative_synth + (1-y_test) * prob_null_synth
    for i in range(n_test):
        U = np.random.rand()
        
        A_sub = np.random.binomial(n=1, p=probs_real[i], size=n_real)
        n_successes_A = np.sum(A_sub)
        B_sub = np.random.binomial(n=1, p=probs_synth[i], size=n_synth)
        n_successes_B = np.sum(B_sub)

        p_real = run_test(n_successes_A, n=n_real, p=0.5, U=U)
        real_pvalues.append(p_real)
        p_synth = run_test(n_successes_B, n=n_synth, p=0.5, U=U)
        synth_pvalues.append(p_synth)
        p_real_synth = run_test(n_successes_A+n_successes_B, n=n_real+n_synth, p=0.5, U=U)
        synth_powered_pvalues.append(p_real_synth)
    return np.array(real_pvalues), np.array(synth_pvalues), np.array(synth_powered_pvalues)


def run_comparison_synthetic(prob_alternative=0.51, prob_null_synth=0.5, prob_alternative_synth=0.51, n_real=50, n_synth=200,
                             alpha=0.05, epsilon=0.02, n_test=1000, p_test=0.05, n_runs=100, seed=42, save_path=None, force_run=False):
    if save_path is not None:
        curr_save_path = save_path + '/' + get_run_description(p_alt=prob_alternative, p_null_synth=prob_null_synth, p_alt_synth=prob_alternative_synth,
                                                               n_real=n_real, n_synth=n_synth, alpha=alpha,
                                                                epsilon=epsilon, n_runs=n_runs, n_test=n_test, p_test=p_test) + '/results'
        os.makedirs(curr_save_path, exist_ok=True)
        # check if this run already saved
        if os.path.exists(curr_save_path + '/results.pkl') and not force_run:
            print('results already exist, skip run ...')
            return
    results = pd.DataFrame({})
    params_dict = {'alpha': alpha, 'epsilon': epsilon, 'n_real': n_real, 'n_synth': n_synth, 'p_alt': prob_alternative,
                   'p_null_synth': prob_null_synth, 'p_alt_synth': prob_alternative_synth,
                   'n_test': n_test, 'p_test': p_test, 'n_runs': n_runs}
    np.random.seed(seed)
    random.seed(seed)
    seed_list = random.sample(range(1, 999999), n_runs)
    n_alternative = int(n_test * p_test)
    test_y = np.concatenate([np.ones((n_alternative,)), np.zeros((n_test - n_alternative,))], axis=0)
    for seed_ in seed_list:
        np.random.seed(seed_)
        random.seed(seed_)
        pvalues_real, pvalues_synth, pvalues_synth_powered = generate_data_and_return_p_values(n_real, n_synth, test_y, prob_alternative, prob_null_synth, prob_alternative_synth)
        
        rejections_synth, _ = BH(pvalues_synth, alpha)
        rejections_synth_real, _ = BH(pvalues_synth_powered, alpha)
        rejections_real, _ = BH(pvalues_real, alpha)
        rejections_real_e, _ = BH(pvalues_real, alpha + epsilon)
        rejections_SynthBH, _ = synth_powered_BH(pvalues_real, pvalues_synth_powered, alpha, epsilon)
        power_synth, _, _, fdr_synth = analyze_performance(rejections_synth, test_y)
        power_synth_real, _, _, fdr_synth_real = analyze_performance(rejections_synth_real, test_y)
        power_real, _, _, fdr_real = analyze_performance(rejections_real, test_y)
        power_real_e, _, _, fdr_real_e = analyze_performance(rejections_real_e, test_y)
        power_SynthBH, _, _, fdr_SynthBH = analyze_performance(rejections_SynthBH, test_y)

        curr_results = pd.DataFrame([{**params_dict, 'Method': 'BH_real', 'Power': power_real, 'FDR': fdr_real},
                                    {**params_dict, 'Method': 'BH_real+e', 'Power': power_real_e, 'FDR': fdr_real_e},
                                    {**params_dict, 'Method': 'BH_synth', 'Power': power_synth, 'FDR': fdr_synth},
                                    {**params_dict, 'Method': 'BH_pooled', 'Power': power_synth_real, 'FDR': fdr_synth_real},
                                    {**params_dict, 'Method': 'SynthBH', 'Power': power_SynthBH, 'FDR': fdr_SynthBH},
                                    ])
        results = pd.concat([results, curr_results])
    if save_path is not None:
        results.to_pickle(curr_save_path + '/results.pkl')
    return results

# FDR experiments

In [ ]:

#### non-null_Real - as a function of n
# parameters
results_dir = './results/'
plots_dir = './plots/'
n_real=500
n_synth=1000
prob_alternative=0.6
prob_null_synth=0.5
prob_alternative_synth=0.55
alpha=0.1
epsilon=0.1
n_test=1000
p_test=0.05
n_runs=100
seed=42

for n_real in [50, 100, 200, 500]:
    run_comparison_synthetic(prob_alternative=prob_alternative, prob_null_synth=prob_null_synth, prob_alternative_synth=prob_alternative_synth,
                             n_real=n_real, n_synth=n_synth, alpha=alpha, epsilon=epsilon, n_test=n_test, p_test=p_test,
                             n_runs=n_runs, seed=seed, save_path=results_dir)

load_and_plot(plots_dir, results_dir, y='Power', x='n_real', alpha=alpha, epsilon=epsilon, n_synth=n_synth, p_alt=prob_alternative, p_alt_synth=prob_alternative_synth, p_null_synth=prob_null_synth,
              n_test=n_test, p_test=p_test, n_runs=n_runs, methods2plot=['BH_real', 'BH_pooled', 'SynthBH', 'BH_real+e'])

load_and_plot(plots_dir, results_dir, y='FDR', x='n_real', alpha=alpha, epsilon=epsilon, n_synth=n_synth, p_alt=prob_alternative, p_alt_synth=prob_alternative_synth, p_null_synth=prob_null_synth,
              n_test=n_test, p_test=p_test, n_runs=n_runs, methods2plot=['BH_real', 'BH_pooled', 'SynthBH', 'BH_real+e'])

In [ ]:

#### non-null_Real - as a function of synthetic signal
# parameters
results_dir = './results/'
plots_dir = './plots/'
n_real=200
n_synth=1000
prob_alternative=0.6
prob_null_synth=0.5
prob_alternative_synth=0.55
alpha=0.1
epsilon=0.1
n_test=1000
p_test=0.05
n_runs=100
seed=42

for prob_alternative_synth in [0.51, 0.53, 0.55, 0.58, 0.6]:
    run_comparison_synthetic(prob_alternative=prob_alternative, prob_null_synth=prob_null_synth, prob_alternative_synth=prob_alternative_synth,
                             n_real=n_real, n_synth=n_synth, alpha=alpha, epsilon=epsilon, n_test=n_test, p_test=p_test,
                             n_runs=n_runs, seed=seed, save_path=results_dir)

load_and_plot(plots_dir, results_dir, y='Power', x='p_alt_synth', alpha=alpha, epsilon=epsilon, n_synth=n_synth, p_alt=prob_alternative, n_real=n_real, p_null_synth=prob_null_synth,
              n_test=n_test, p_test=p_test, n_runs=n_runs, methods2plot=['BH_real', 'BH_pooled', 'SynthBH', 'BH_real+e'])

load_and_plot(plots_dir, results_dir, y='FDR', x='p_alt_synth', alpha=alpha, epsilon=epsilon, n_synth=n_synth, p_alt=prob_alternative, n_real=n_real, p_null_synth=prob_null_synth,
              n_test=n_test, p_test=p_test, n_runs=n_runs, methods2plot=['BH_real', 'BH_pooled', 'SynthBH', 'BH_real+e'])

In [ ]:

#### non-null_Real - as a function of synthetic signal
# parameters
results_dir = './results/'
plots_dir = './plots/'
n_real=200
n_synth=1000
prob_alternative=0.6
prob_null_synth=None
prob_alternative_synth=0.55
alpha=0.1
epsilon=0.1
n_test=1000
p_test=0.05
n_runs=100
seed=42

for prob_alternative_synth in [0.51, 0.53, 0.55, 0.58, 0.6]:
    run_comparison_synthetic(prob_alternative=prob_alternative, prob_null_synth=prob_alternative_synth, prob_alternative_synth=prob_alternative_synth,
                             n_real=n_real, n_synth=n_synth, alpha=alpha, epsilon=epsilon, n_test=n_test, p_test=p_test,
                             n_runs=n_runs, seed=seed, save_path=results_dir)

load_and_plot(plots_dir, results_dir, y='Power', x='p_alt_synth', alpha=alpha, epsilon=epsilon, n_synth=n_synth, p_alt=prob_alternative, n_real=n_real,
              n_test=n_test, p_test=p_test, n_runs=n_runs, methods2plot=['BH_real', 'BH_pooled', 'SynthBH', 'BH_real+e'])

load_and_plot(plots_dir, results_dir, y='FDR', x='p_alt_synth', alpha=alpha, epsilon=epsilon, n_synth=n_synth, p_alt=prob_alternative, n_real=n_real,
              n_test=n_test, p_test=p_test, n_runs=n_runs, methods2plot=['BH_real', 'BH_pooled', 'SynthBH', 'BH_real+e'])

In [ ]:

#### non-null_Real - as a function of epsilon
# parameters
results_dir = './results/'
plots_dir = './plots/'
n_real=200
n_synth=1000
prob_alternative=0.6
prob_null_synth=0.5
prob_alternative_synth=0.55
alpha=0.1
epsilon=0.1
n_test=1000
p_test=0.05
n_runs=100
seed=42

for epsilon in [0.02, 0.05, 0.1, 0.15, 0.2]:
    run_comparison_synthetic(prob_alternative=prob_alternative, prob_null_synth=prob_null_synth, prob_alternative_synth=prob_alternative_synth,
                             n_real=n_real, n_synth=n_synth, alpha=alpha, epsilon=epsilon, n_test=n_test, p_test=p_test,
                             n_runs=n_runs, seed=seed, save_path=results_dir)

load_and_plot(plots_dir, results_dir, y='Power', x='epsilon', alpha=alpha, n_synth=n_synth, p_alt=prob_alternative,
              p_alt_synth=prob_alternative_synth, n_real=n_real, p_null_synth=prob_null_synth,
              n_test=n_test, p_test=p_test, n_runs=n_runs, methods2plot=['BH_real', 'BH_pooled', 'SynthBH', 'BH_real+e'])

load_and_plot(plots_dir, results_dir, y='FDR', x='epsilon', alpha=alpha, n_synth=n_synth, p_alt=prob_alternative,
              p_alt_synth=prob_alternative_synth, n_real=n_real, p_null_synth=prob_null_synth,
              n_test=n_test, p_test=p_test, n_runs=n_runs, methods2plot=['BH_real', 'BH_pooled', 'SynthBH', 'BH_real+e'])

In [ ]:

#### non-null_Real - as a function of epsilon
# parameters
results_dir = './results/'
plots_dir = './plots/'
n_real=200
n_synth=1000
prob_alternative=0.6
prob_null_synth=None
prob_alternative_synth=0.55
alpha=0.1
epsilon=0.1
n_test=1000
p_test=0.05
n_runs=100
seed=42

for epsilon in [0.02, 0.05, 0.1, 0.15, 0.2]:
    run_comparison_synthetic(prob_alternative=prob_alternative, prob_null_synth=prob_alternative_synth, prob_alternative_synth=prob_alternative_synth,
                             n_real=n_real, n_synth=n_synth, alpha=alpha, epsilon=epsilon, n_test=n_test, p_test=p_test,
                             n_runs=n_runs, seed=seed, save_path=results_dir)

load_and_plot(plots_dir, results_dir, y='Power', x='epsilon', alpha=alpha, n_synth=n_synth, p_alt=prob_alternative,
              p_alt_synth=prob_alternative_synth, n_real=n_real, p_null_synth=prob_alternative_synth,
              n_test=n_test, p_test=p_test, n_runs=n_runs, methods2plot=['BH_real', 'BH_pooled', 'SynthBH', 'BH_real+e'])

load_and_plot(plots_dir, results_dir, y='FDR', x='epsilon', alpha=alpha, n_synth=n_synth, p_alt=prob_alternative,
              p_alt_synth=prob_alternative_synth, n_real=n_real, p_null_synth=prob_alternative_synth,
              n_test=n_test, p_test=p_test, n_runs=n_runs, methods2plot=['BH_real', 'BH_pooled', 'SynthBH', 'BH_real+e'])